# Milestone 1 – Data Pipeline & Baseline Model
**Student:** Fatima Tangatarova (TAN0075)  
**Task:** classify news headlines into one of 10 selected categories.

The baseline uses a single-layer GRU without pretrained weights. Run all cells to obtain results for this version. The hyperparameter changes are experiments to evaluate, not a guarantee of better validation accuracy.


## Environment
This notebook requires Python, PyTorch, numpy, pandas, matplotlib, seaborn, and scikit-learn. Place `News_Category_Dataset_v3.json` next to the notebook or update `DATA_PATH`. Tokenization uses a regular expression, so TextBlob and NLTK data downloads are not required.


In [ ]:
from sklearn.model_selection import train_test_split
import unicodedata, re, string, itertools
from collections import Counter
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
sns.set_theme(style='whitegrid')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch version:', torch.__version__)
print('device:', device)


In [ ]:
from pathlib import Path
DATA_PATH = Path('News_Category_Dataset_v3.json')
if not DATA_PATH.exists():
    candidates = [Path('/content/News_Category_Dataset_v3.json'), Path.home() / 'Downloads/News_Category_Dataset_v3.json']
    DATA_PATH = next((p for p in candidates if p.exists()), DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError('Set DATA_PATH to the News_Category_Dataset_v3.json file.')
df = pd.read_json(DATA_PATH, lines=True)
print(df.shape)
df.head()


## Selecting 10 Categories
For the initial baseline, we simplify the task to the 10 most frequent categories: POLITICS, WELLNESS, ENTERTAINMENT, TRAVEL, STYLE & BEAUTY, PARENTING, HEALTHY LIVING, QUEER VOICES, FOOD & DRINK, and BUSINESS.

We sample up to 5,000 articles per category. This produces an approximately balanced dataset of up to 50,000 headlines, with more training examples per class than in the original version. Reducing the number of categories changes the task; it does not prove that overfitting has been resolved. WELLNESS and HEALTHY LIVING may overlap in topic, so potential confusion between them should be checked in the results.


In [ ]:
print('Number of categories:', df['category'].nunique())
print('\nAvailable categories:')
print(df['category'].value_counts())


We remove missing and empty headlines. To detect duplicates, we lowercase the headlines and normalize whitespace. If the same normalized headline has multiple category labels, we remove it as ambiguous. For other duplicates, we keep one record to prevent identical headlines from appearing across the training, validation, and test sets.


In [ ]:
SELECTED_CATEGORIES = [
    "POLITICS", "WELLNESS", "ENTERTAINMENT", "TRAVEL",
    "STYLE & BEAUTY", "PARENTING", "HEALTHY LIVING", "QUEER VOICES",
    "FOOD & DRINK", "BUSINESS"
]
MAX_PER_CATEGORY = 5000

df = df[["headline", "category"]].dropna().copy()
df = df[df["headline"].str.strip() != ""].copy()
df["headline_key"] = df["headline"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
label_counts = df.groupby("headline_key")["category"].transform("nunique")
df = df[label_counts == 1].drop_duplicates(subset="headline_key")
df = df[df["category"].isin(SELECTED_CATEGORIES)].copy()
assert set(df["category"].unique()) == set(SELECTED_CATEGORIES)

df = pd.concat(
    [group.sample(n=min(MAX_PER_CATEGORY, len(group)), random_state=SEED)
     for _, group in df.groupby("category")],
    ignore_index=True
)
print("Dataset shape:", df.shape)
print("Number of selected categories:", df["category"].nunique())
print(df["category"].value_counts())



## Labels and Data Splitting

We convert category names into numerical labels and split
the dataset into 70% training, 20% validation, and 10% testing data.


In [ ]:
categories = sorted(df['category'].unique())
label_to_id = {category: i for i, category in enumerate(categories)}
df['label'] = df['category'].map(label_to_id)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=1 / 3, random_state=42, stratify=temp_df['label'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print('Number of classes:', len(categories))
print('Training samples:', len(train_df))
print('Validation samples:', len(val_df))
print('Testing samples:', len(test_df))

assert set(train_df["headline_key"]).isdisjoint(val_df["headline_key"])
assert set(train_df["headline_key"]).isdisjoint(test_df["headline_key"])
assert set(val_df["headline_key"]).isdisjoint(test_df["headline_key"])


## Tokenization and Vocabulary
We lowercase headlines and split them into words using a regular expression, preserving apostrophes within words. Subsequent normalization removes accents, digits, and leading or trailing punctuation. The vocabulary is built from training data only. This is a simple baseline tokenizer. Because it replaces the original TextBlob tokenizer, the new results do not isolate the effect of regularization alone.


In [ ]:
def form_sentence(tweet):
    return re.findall("[^\\W\\d_]+(?:'[^\\W\\d_]+)*", tweet.lower().replace('’', "'"))

def remove_non_ascii(words):
    out = []
    for w in words:
        w = unicodedata.normalize('NFKD', w).encode('ascii', 'ignore').decode('utf-8', 'ignore')
        out.append(w)
    return out

def to_lowercase(words):
    return [w.lower() for w in words]

def remove_numbers(words):
    out = []
    for w in words:
        w = re.sub('\\d+', '', w)
        if w != '':
            out.append(w)
    return out

def remove_punctuation(words):
    out = []
    for w in words:
        w = w.strip(string.punctuation)
        if w != '':
            out.append(w)
    return out

def normalize(words):
    words = remove_non_ascii(words)
    words = to_lowercase(words)
    words = remove_numbers(words)
    words = remove_punctuation(words)
    return words

def fix_nt(words):
    res = []
    for w in words:
        if w in ("n't", 'nt') and res:
            res[-1] = res[-1] + "n't"
        else:
            res.append(w)
    return res


We apply the same transformations to the training, validation, and test sets. Validation and test data do not contribute new words to the vocabulary.


In [ ]:
train_df['Words'] = train_df['headline'].apply(form_sentence)
train_df.loc[:5, ['headline', 'Words']]
train_df['Words'] = train_df['Words'].apply(normalize)
train_df.loc[:5, ['headline', 'Words']]
train_df['Words'] = train_df['Words'].apply(fix_nt)
train_df.loc[:5, ['headline', 'Words']]
for data in [val_df, test_df]:
    data['Words'] = data['headline'].apply(form_sentence)
    data['Words'] = data['Words'].apply(normalize)
    data['Words'] = data['Words'].apply(fix_nt)


## Input and Target
We use the news headline as the input and its category as the target label. Headlines are short, which keeps baseline training relatively fast. Model parameters are learned from the training set only.

- PAD = padding token
- UNK = unknown token


In [ ]:
word_counts = Counter()
for words in train_df['Words']:
    word_counts.update(words)
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.items():
    if count >= 2:
        vocab[word] = len(vocab)
print('Vocabulary size:', len(vocab))
list(vocab.items())[:20]


## Numerical Encoding

We convert each token into its vocabulary index.
Unknown words are mapped to the `<UNK>` token.

In [ ]:
def words_to_indices(words):
    return [vocab.get(word, vocab['<UNK>']) for word in words]
train_df['Indices'] = train_df['Words'].apply(words_to_indices)
val_df['Indices'] = val_df['Words'].apply(words_to_indices)
test_df['Indices'] = test_df['Words'].apply(words_to_indices)
train_df.loc[:5, ['headline', 'Words', 'Indices']]


## Custom Dataset

The custom `NewsDataset` stores each encoded headline as a tensor
and returns one sequence together with its class label.

`__len__` returns the number of samples and `__getitem__`
returns one sample by index.

In [ ]:
PAD_IDX = vocab['<PAD>']

class NewsDataset(Dataset):

    def __init__(self, dataframe):
        self.sequences = [torch.tensor(indices, dtype=torch.long) for indices in dataframe['Indices']]
        self.labels = torch.tensor(dataframe['label'].values, dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, i):
        return (self.sequences[i], self.labels[i])


## Batch Padding

Headlines have different lengths, so they cannot be stacked directly
into one batch tensor.

The `collate_fn` pads shorter sequences with `<PAD>` tokens and stores
their original lengths.

In [ ]:
def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    labels = torch.stack(labels)
    return (padded, lengths, labels)


## Removing Empty Sequences

We remove empty token sequences because `pack_padded_sequence`
cannot process sequences with length zero.

In [ ]:
train_df = train_df[train_df['Indices'].apply(len) > 0].reset_index(drop=True)
val_df = val_df[val_df['Indices'].apply(len) > 0].reset_index(drop=True)
test_df = test_df[test_df['Indices'].apply(len) > 0].reset_index(drop=True)


## DataLoaders

We create separate DataLoaders for training, validation, and testing.

The training loader uses shuffling, while validation and test loaders
keep a fixed order.

In [ ]:
BATCH_SIZE = 128
train_ds = NewsDataset(train_df)
val_ds = NewsDataset(val_df)
test_ds = NewsDataset(test_df)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)


## Batch Check

We inspect one training batch to verify the tensor shapes,
sequence lengths, and labels before building the model.

In [ ]:
xb, lengths, yb = next(iter(train_loader))
print('Padded batch:', xb.shape)
print('Lengths:', lengths.shape)
print('Labels:', yb.shape)
print('\nFirst sequence length:', lengths[0].item())
print('First label:', yb[0].item())


## Decoded Batch

We decode one transformed batch back into words to verify
that tokenization, numerical encoding, and padding work correctly.

In [ ]:
id_to_label = {v: k for k, v in label_to_id.items()}
id_to_word = {index: word for word, index in vocab.items()}
xb, lengths, yb = next(iter(train_loader))
for i in range(3):
    real_length = lengths[i].item()
    decoded_words = [id_to_word[index.item()] for index in xb[i][:real_length]]
    print('Headline:', ' '.join(decoded_words))
    print('Category:', id_to_label[yb[i].item()])
    print('Tensor shape:', xb[i].shape)
    print('Real length:', real_length)
    print()


## Baseline GRU Model

The model converts word indices into embeddings and processes
the sequence with a GRU.

The final hidden state is passed to a linear layer to predict
one of the news categories.

In [ ]:
class GRUClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        embedded = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        hidden = hidden[-1]
        hidden = self.dropout(hidden)
        output = self.fc(hidden)
        return output


## Model Size and Regularization
We reduce the embedding dimension from 128 to 64 and keep 64 GRU hidden units. With the original vocabulary of 14,046 words, the embedding alone had 1,797,888 parameters; halving its dimension halves that count for the same vocabulary. We apply dropout with probability 0.3 to both the embeddings and the final hidden state, using a separate `nn.Dropout` module with the single-layer GRU.

These settings are a starting point. We assess their benefit using validation results, not the test set.


In [ ]:
EMBED_DIM = 64
HIDDEN_DIM = 64
NUM_CLASSES = len(categories)
model = GRUClassifier(vocab_size=len(vocab), embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(device)
print(model)


## Output Shape Check
We verify that the model returns one logit per category for each sample in the batch.


In [ ]:
xb, lengths, yb = next(iter(train_loader))
xb = xb.to(device)
yb = yb.to(device)
outputs = model(xb, lengths)
print('Input shape:', xb.shape)
print('Output shape:', outputs.shape)


## Explicit Training and Validation Loops
We use `CrossEntropyLoss`: model outputs have shape `[batch, 10]`, while labels have shape `[batch]` and dtype `torch.long`. We do not apply sigmoid or softmax before the loss function.

AdamW uses a learning rate of 0.001 and weight decay of 0.001. Early stopping ends training after three epochs without a decrease in validation loss. We save and restore the weights with the lowest validation loss. This criterion may select a different epoch than the one with the highest accuracy. Metrics are accumulated across batches, accounting for batch size.


In [ ]:
torch.manual_seed(42)
model = GRUClassifier(vocab_size=len(vocab), embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, num_classes=NUM_CLASSES, pad_idx=PAD_IDX).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.001)


In [ ]:
from copy import deepcopy
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val = float('inf')
best_state = None
best_epoch = 0
patience = 3
bad_epochs = 0


In [ ]:
N_EPOCHS = 30
for epoch in range(N_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for xb, lengths, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb, lengths)
        assert logits.shape == (yb.size(0), NUM_CLASSES)
        assert yb.ndim == 1 and yb.dtype == torch.long
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        predictions = logits.argmax(dim=1)
        correct += (predictions == yb).sum().item()
        total += yb.size(0)
    train_loss = running_loss / total
    train_acc = correct / total
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, lengths, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb, lengths)
            loss = criterion(logits, yb)
            running_loss += loss.item() * xb.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == yb).sum().item()
            total += yb.size(0)
    val_loss = running_loss / total
    val_acc = correct / total
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    if val_loss < best_val:
        best_val = val_loss
        best_state = deepcopy(model.state_dict())
        best_epoch = epoch + 1
        bad_epochs = 0
    else:
        bad_epochs += 1
    print(f'Epoch {epoch + 1:2d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
    if bad_epochs >= patience:
        print(f'Early stopping: {patience} epochs without improvement in validation loss.')
        break
model.load_state_dict(best_state)
torch.save(model.state_dict(), 'best_model.pt')
print(f'Restored model from epoch {best_epoch}, val loss = {best_val:.4f}')


## Training History

We plot training and validation loss and accuracy over epochs.
The plots help us identify overfitting and the best training point.

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(8, 5))
plt.plot(epochs, history['train_loss'], label='Train Loss')
plt.plot(epochs, history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs, history['train_acc'], label='Train Accuracy')
plt.plot(epochs, history['val_acc'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid()
plt.show()


## Baseline Interpretation
In the **original saved run**, validation loss reached its minimum of 2.8877 at epoch 12 (training accuracy: 38.51%; validation accuracy: 24.68%). At epoch 30, training accuracy was 67.35%, validation accuracy was 27.42%, and validation loss was 3.5130. Increasing validation loss alongside decreasing training loss indicates overfitting. Accuracy can still improve slightly because the loss also reflects how confident incorrect predictions are.

These numbers are not results for the revised model. After running this version, examine the best epoch, validation loss, accuracy, and macro-F1. The training–validation gap may not disappear. A correct pipeline and an honest interpretation are central to this milestone.

The current version uses headlines only and up to 5,000 examples per category, giving approximately 3,500 training examples per category with the 70/20/10 split. The original run used 42 categories, so its accuracy cannot be directly compared with the new 10-category task. Related categories may make classification harder; a confusion matrix can help investigate this. Further experiments could use more training data or include `short_description`. Category merging would require a clear justification.


## Overfit Sanity Check
We train a separate model on 12 training samples. Here, overfitting is intentional: the targets are 100% accuracy and a loss below 0.01. We disable dropout and use no weight decay. After updating the weights, we evaluate all 12 samples in `eval()` mode with `no_grad()`. This model does not replace the baseline and is not used to measure generalization.


In [ ]:
small_df = train_df.iloc[:12].copy()
small_ds = NewsDataset(small_df)
small_loader = DataLoader(small_ds, batch_size=12, shuffle=True, num_workers=0, collate_fn=collate_fn)


In [ ]:
torch.manual_seed(42)
overfit_model = GRUClassifier(vocab_size=len(vocab), embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, num_classes=NUM_CLASSES, pad_idx=PAD_IDX, dropout=0.0).to(device)
overfit_criterion = nn.CrossEntropyLoss()
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.01)


In [ ]:
overfit_losses = []
overfit_accuracy = 0.0
for epoch in range(300):
    overfit_model.train()
    for xb, lengths, yb in small_loader:
        xb, yb = (xb.to(device), yb.to(device))
        overfit_optimizer.zero_grad()
        logits = overfit_model(xb, lengths)
        assert logits.shape == (yb.size(0), NUM_CLASSES)
        assert yb.ndim == 1 and yb.dtype == torch.long
        loss = overfit_criterion(logits, yb)
        loss.backward()
        overfit_optimizer.step()
    overfit_model.eval()
    total_loss, correct, total = (0.0, 0, 0)
    with torch.no_grad():
        for xb, lengths, yb in small_loader:
            xb, yb = (xb.to(device), yb.to(device))
            logits = overfit_model(xb, lengths)
            total_loss += overfit_criterion(logits, yb).item() * yb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            total += yb.size(0)
    overfit_loss = total_loss / total
    overfit_accuracy = correct / total
    overfit_losses.append(overfit_loss)
    if (epoch + 1) % 20 == 0 or (correct == total and overfit_loss < 0.01):
        print(f'Epoch {epoch + 1}: loss={overfit_loss:.6f}, accuracy={overfit_accuracy:.2%}')
    if correct == total and overfit_loss < 0.01:
        break
sanity_passed = overfit_accuracy == 1.0 and overfit_loss < 0.01
print('Sanity check:', 'PASSED' if sanity_passed else 'FAILED - inspect the pipeline and training progress')
plt.plot(range(1, len(overfit_losses) + 1), overfit_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss on 12 samples after weight updates')
plt.show()


## Restored Baseline Evaluation and Summary
Validation metrics guide configuration selection. The test set is used only after that selection is complete. Macro-F1 gives each class equal weight. The following cell reports the results of the current run. Before submission, add your interpretation: whether the training–validation gap decreased, how accuracy changed, and whether the sanity check met both criteria.


In [ ]:
def evaluate_baseline(loader):
    model.eval()
    total_loss, total = (0.0, 0)
    true_labels, predicted_labels = ([], [])
    with torch.no_grad():
        for xb, lengths, yb in loader:
            xb, yb = (xb.to(device), yb.to(device))
            logits = model(xb, lengths)
            total_loss += criterion(logits, yb).item() * yb.size(0)
            total += yb.size(0)
            true_labels.extend(yb.cpu().tolist())
            predicted_labels.extend(logits.argmax(1).cpu().tolist())
    return (total_loss / total, accuracy_score(true_labels, predicted_labels), f1_score(true_labels, predicted_labels, labels=list(range(NUM_CLASSES)), average='macro', zero_division=0))
val_metrics = evaluate_baseline(val_loader)
print(f'Best epoch: {best_epoch}')
print(f'Validation: loss={val_metrics[0]:.4f}, accuracy={val_metrics[1]:.2%}, macro-F1={val_metrics[2]:.4f}')
print(f'Sanity check: accuracy={overfit_accuracy:.2%}, loss={overfit_loss:.6f}, passed={sanity_passed}')
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    test_metrics = evaluate_baseline(test_loader)
    print(f'Test: loss={test_metrics[0]:.4f}, accuracy={test_metrics[1]:.2%}, macro-F1={test_metrics[2]:.4f}')
